# 10 project rows (g2)

Every project row the parser reads. What you should see: every parsed edition `ok`, the newest edition of all six current lists among them; rows equal the refs counted in the page text; MW within 1 of the printed total, and the energised renewable list at 448 rows and about 2,776 MW against the summary report's grid-scale line. Then the hand count: open two pages of each PDF and check the rows, first ref and last ref.

In [ ]:
import subprocess

from iegrid import store
from iegrid.checks import by_technology, hand_count_pages, latest_parsed, ties, unparsed

snap = store.latest_snapshot()
sha = subprocess.run(["git", "-C", str(store.ROOT), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, check=True).stdout.strip()
con = store.connect_read_only(snap)
print(f"snapshot {snap.name}  git {sha}")

## done-when: every list ties

In [ ]:
found = ties(con)
for t in found:
    target = "-" if t.target_mw is None else f"{t.target_mw}" + ("" if t.target_rows is None else f" / {t.target_rows} rows")
    print(f"{t.source_id:32} {t.as_of}  rows {t.rows:4} text {t.text_refs:4}  parsed {t.parsed_mw:>11} MW  "
          f"vs {t.against}: {target}  ok {t.ok}")
missing = unparsed(con)
assert not missing, f"stored but not parsed (run iegrid.load): {missing}"
from iegrid.parse import PARSERS

assert set(PARSERS) <= set(latest_parsed(con)), "a current list's newest edition is not parsed"
assert all(t.ok for t in found), "a list does not tie"
print(f"\n{len(found)} parsed editions, {sum(t.rows for t in found)} rows, all tie")

## hand count: first two pages with rows in each list

In [ ]:
print(hand_count_pages(con))

## what is in the lists

In [ ]:
print(by_technology(con))

## exhibit: what is waiting to connect

In [ ]:
from iegrid import exhibits

fig = exhibits.contracted_by_technology(con)
exhibits.save(fig, "g2_contracted_by_technology")